In [0]:
from pyspark.sql import functions as F

# Load the results and KenPom feature tables
results = spark.table("workspace.marchmadness.silver_kaggle")
kenpom = spark.table("workspace.marchmadness.silver_kenpom")
display(results)
display(kenpom)

In [0]:
# Join KenPom features for Team 1
train_team1 = results.join(kenpom, results.Team1Season_Team == kenpom.Season_Team, "left")
train_team1 = train_team1.withColumnRenamed("Seed", "Seed_x")\
    .withColumnRenamed("Wins", "Wins_x")\
    .withColumnRenamed("AdjEM", "AdjEM_x")\
    .withColumnRenamed("AdjO", "AdjO_x")\
    .withColumnRenamed("AdjD", "AdjD_x")\
    .withColumnRenamed("SoS_AdjEM", "SoS_AdjEM_x")\
    .withColumnRenamed("OppO", "OppO_x")\
    .withColumnRenamed("OppD", "OppD_x")\
    .withColumnRenamed("TeamID", "TeamID_1")\
    .withColumnRenamed("TeamName_clean", "TeamName_1")
display(train_team1)

In [0]:
# Join KenPom features for Team 2
train = train_team1.join(kenpom, train_team1.Team2Season_Team == kenpom.Season_Team, "left")
train = train.withColumnRenamed("Seed", "Seed_y")\
    .withColumnRenamed("Wins", "Wins_y")\
    .withColumnRenamed("AdjEM", "AdjEM_y")\
    .withColumnRenamed("AdjO", "AdjO_y")\
    .withColumnRenamed("AdjD", "AdjD_y")\
    .withColumnRenamed("SoS_AdjEM", "SoS_AdjEM_y")\
    .withColumnRenamed("OppO", "OppO_y")\
    .withColumnRenamed("OppD", "OppD_y")\
    .withColumnRenamed("TeamID", "TeamID_2")\
    .withColumnRenamed("TeamName_clean", "TeamName_2")
display(train)

In [0]:
# Feature engineering for ML training
train = train.withColumn("SeedDelta", -(F.col("Seed_x" ).cast("int") - F.col("Seed_y").cast("int")))\
    .withColumn("WinDelta", F.col("Wins_x").cast("int") - F.col("Wins_y").cast("int"))\
    .withColumn("AdjEMDelta", F.col("AdjEM_x").cast("float") - F.col("AdjEM_y").cast("float"))\
    .withColumn("SoS_AdjEMDelta", F.col("SoS_AdjEM_x").cast("float") - F.col("SoS_AdjEM_y").cast("float"))
display(train)

In [0]:
# Select ML columns and save to gold table
train_ml = train.select(
    "id", "Team1Wins", "SeedDelta", "WinDelta", "AdjEMDelta", "SoS_AdjEMDelta",
    "Seed_x", "Wins_x", "AdjEM_x", "AdjO_x", "AdjD_x", "SoS_AdjEM_x", "OppO_x", "OppD_x",
    "Seed_y", "Wins_y", "AdjEM_y", "AdjO_y", "AdjD_y", "SoS_AdjEM_y", "OppO_y", "OppD_y"
)
train_ml.write.format("delta").mode("overwrite").saveAsTable("marchmadness.gold_trainingdata")
display(train_ml)

In [0]:
# Select the relevant ML feature columns
target_cols = [
    "id", "Team1Wins", "SeedDelta", "WinDelta", "AdjEMDelta", "SoS_AdjEMDelta",
    "Seed_x", "Wins_x", "AdjEM_x", "AdjO_x", "AdjD_x", "SoS_AdjEM_x", "OppO_x", "OppD_x",
    "Seed_y", "Wins_y", "AdjEM_y", "AdjO_y", "AdjD_y", "SoS_AdjEM_y", "OppO_y", "OppD_y"
]
train_ml = train.select(*target_cols)
display(train_ml)

In [0]:
# Save the training data.
train_ml.write.format("delta").mode("overwrite").saveAsTable("marchmadness.gold_trainingdata")